In [ ]:
# C12c: IT-Oriented Re-verification — Chronic-Specific vs Normal Hepatitis Response
# Key question: Which findings are chronic HBV-specific (IT/IA ≠ AR) vs general hepatitis (IA ≈ AR)?
# Run after C12/C12b notebooks (uses obs, safe_clonality)

In [8]:
# ============================================================
# CELL 0: GPU SETUP & VERIFICATION
# ============================================================
!pip install scanpy anndata matplotlib seaborn scipy -q

import subprocess
import sys

print("=" * 70)
print("  V18 C3: GPU ENVIRONMENT SETUP")
print("=" * 70)

# GPU detection
try:
    result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,compute_cap',
                            '--format=csv,noheader'], capture_output=True, text=True)
    gpu_info = result.stdout.strip()
    print(f"✅ GPU detected: {gpu_info}")
except:
    print("⚠️ No GPU detected — will use CPU fallback")

# Install CuPy for GPU acceleration
try:
    import cupy as cp
    print(f"✅ CuPy {cp.__version__} ready")
    print(f"   GPU memory: {cp.cuda.Device(0).mem_info[1] / 1e9:.1f} GB total")
except ImportError:
    print("📦 Installing CuPy...")
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'cupy-cuda12x', '-q'])
    import cupy as cp
    print(f"✅ CuPy {cp.__version__} installed")

  V18 C3: GPU ENVIRONMENT SETUP
✅ GPU detected: NVIDIA A100-SXM4-80GB, 81920 MiB, 8.0
✅ CuPy 14.0.1 ready
   GPU memory: 85.1 GB total


In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
from scipy.stats import mannwhitneyu
import numpy as np, pandas as pd
import warnings; warnings.filterwarnings('ignore')

# Ensure obs and functions are available
try:
    _ = obs['donor']
except:
    import scanpy as sc
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_PATH = '/content/drive/MyDrive/ITLAS/data/processed/GSE182159_gut2021_annotated.h5ad'
    adata = sc.read_h5ad(DATA_PATH, backed='r')
    obs = adata.obs.copy()
    obs['donor'] = obs['sample'].astype(str).str.split('_').str[1]

def safe_clonality(cc):
    nu=len(cc); nt=cc.sum()
    if nu<=0 or nt<=0: return 0.0
    if nu==1: return 1.0 if nt>1 else 0.0
    fr=cc.values/nt; fr=fr[fr>0]
    ent=-np.sum(fr*np.log2(fr))
    return 1-(ent/np.log2(nu)) if np.log2(nu)>0 else 0.0

def mw(a, b, label=''):
    a=a.dropna(); b=b.dropna()
    if len(a)<2 or len(b)<2: return None
    stat,p = mannwhitneyu(a, b, alternative='two-sided')
    am,bm = a.mean(),b.mean()
    d = '↑' if bm>am else '↓'
    pct = ((bm-am)/am*100) if am!=0 else float('inf')
    pt=0; pc=0
    for av in a:
        for bv in b:
            pt+=1
            if (bm>am and bv>av) or (bm<=am and bv<=av): pc+=1
    sig = '★' if p<0.05 else '†' if p<0.10 else ' '
    return {'label':label,'A_m':am,'B_m':bm,'d':d,'pct':pct,'p':p,'cons':f'{pc}/{pt}','sig':sig,'nA':len(a),'nB':len(b)}

In [11]:
# ============================================================
# PART 1: B/PlasmaB SUBCLUSTER PROPORTIONS — Full Pairwise
# ============================================================
print('='*70)
print('PART 1: Liver B/PlasmaB Subcluster — Chronic-Specific Analysis')
print('  Key: IA≈AR → normal hepatitis response')
print('       IT/IA ≠ AR → chronic HBV-specific')
print('='*70)

bp = obs[obs['major_lineage'].isin(['B','PlasmaB'])].copy()
sc_list = sorted(bp['gut2021_subcluster_v2'].unique())

sc_rows = []
for (stage, tissue, donor), grp in bp.groupby(['Stage','tissue','donor'], observed=True):
    total = len(grp)
    if total < 5: continue
    counts = grp['gut2021_subcluster_v2'].value_counts()
    for sc in sc_list:
        sc_rows.append({'Stage':stage,'tissue':tissue,'donor':donor,
                        'subcluster':sc,'proportion':counts.get(sc,0)/total*100})
sc_df = pd.DataFrame(sc_rows)

key_subclusters = ['B_c04-COCH', 'B_c07-FCRL5', 'plasmaB_c01-SDC1', 'plasmaB_c03-MKI67']
comparisons = [('NL','IT'), ('NL','IA'), ('NL','AR'), ('IT','IA'), ('IT','AR'), ('IA','AR')]

for sc in key_subclusters:
    print(f'\n{"─"*70}')
    print(f'  {sc} — LIVER')
    print(f'{"─"*70}')
    sub = sc_df[(sc_df['tissue']=='Liver') & (sc_df['subcluster']==sc)]

    # Show individual donor values
    for stage in ['NL','IT','IA','AR','CR']:
        s = sub[sub['Stage']==stage]
        if len(s)==0: continue
        vals = sorted(s['proportion'].values, reverse=True)
        print(f'  {stage} (n={len(s)}): mean={s.proportion.mean():.1f}%, '
              f'donors=[{", ".join(f"{v:.1f}" for v in vals)}]')

    # All pairwise
    print(f'\n  Pairwise comparisons:')
    for s1, s2 in comparisons:
        a = sub[sub['Stage']==s1]['proportion']
        b = sub[sub['Stage']==s2]['proportion']
        r = mw(a, b, f'{s1}→{s2}')
        if r:
            print(f'    {r["sig"]} {s1}→{s2}: {r["A_m"]:.1f}%→{r["B_m"]:.1f}% '
                  f'({r["d"]}{abs(r["pct"]):.0f}%) p={r["p"]:.4f} [{r["cons"]}]')
        else:
            print(f'      {s1}→{s2}: insufficient data')

    # Chronic-specific test: (IT+IA) vs AR
    chronic = sub[sub['Stage'].isin(['IT','IA'])]['proportion']
    ar = sub[sub['Stage']=='AR']['proportion']
    r_chr = mw(chronic, ar, '(IT+IA) vs AR')
    if r_chr:
        print(f'\n    → (IT+IA) vs AR: {r_chr["A_m"]:.1f}%→{r_chr["B_m"]:.1f}% '
              f'({r_chr["d"]}{abs(r_chr["pct"]):.0f}%) p={r_chr["p"]:.4f} [{r_chr["cons"]}]')

    # Classification
    nl_it = mw(sub[sub['Stage']=='NL']['proportion'], sub[sub['Stage']=='IT']['proportion'])
    ia_ar = mw(sub[sub['Stage']=='IA']['proportion'], sub[sub['Stage']=='AR']['proportion'])
    it_ar = mw(sub[sub['Stage']=='IT']['proportion'], sub[sub['Stage']=='AR']['proportion'])

    nl_it_sig = nl_it and nl_it['p'] < 0.10
    ia_ar_sig = ia_ar and ia_ar['p'] < 0.10
    it_ar_sig = it_ar and it_ar['p'] < 0.10

    if nl_it_sig and not ia_ar_sig:
        interpret = '⚠️ NL→IT sig BUT IA≈AR → may be general hepatitis response'
    elif nl_it_sig and ia_ar_sig:
        interpret = '🔥 NL→IT sig AND IA≠AR → CHRONIC HBV-SPECIFIC'
    elif nl_it_sig and it_ar_sig:
        interpret = '🔥 NL→IT sig AND IT≠AR → CHRONIC HBV-SPECIFIC (IT level)'
    else:
        interpret = '  NL→IT NS'
    print(f'\n    INTERPRETATION: {interpret}')

PART 1: Liver B/PlasmaB Subcluster — Chronic-Specific Analysis
  Key: IA≈AR → normal hepatitis response
       IT/IA ≠ AR → chronic HBV-specific

──────────────────────────────────────────────────────────────────────
  B_c04-COCH — LIVER
──────────────────────────────────────────────────────────────────────
  NL (n=6): mean=1.3%, donors=[4.0, 1.7, 1.6, 0.6, 0.0, 0.0]
  IT (n=5): mean=7.1%, donors=[12.1, 8.0, 6.3, 5.2, 3.7]
  IA (n=5): mean=10.7%, donors=[29.4, 9.2, 8.3, 3.7, 2.9]
  AR (n=3): mean=12.8%, donors=[18.5, 13.4, 6.4]
  CR (n=3): mean=10.9%, donors=[12.6, 10.4, 9.6]

  Pairwise comparisons:
    ★ NL→IT: 1.3%→7.1% (↑438%) p=0.0135 [29/30]
    ★ NL→IA: 1.3%→10.7% (↑717%) p=0.0222 [28/30]
    ★ NL→AR: 1.3%→12.8% (↑874%) p=0.0275 [18/18]
      IT→IA: 7.1%→10.7% (↑52%) p=0.8413 [14/25]
      IT→AR: 7.1%→12.8% (↑81%) p=0.1429 [13/15]
      IA→AR: 10.7%→12.8% (↑19%) p=0.5714 [10/15]

    → (IT+IA) vs AR: 8.9%→12.8% (↑44%) p=0.2168 [23/30]

    INTERPRETATION: ⚠️ NL→IT sig BUT IA≈AR 

In [12]:
# ============================================================
# PART 2: BCR METRICS — Chronic-Specific Analysis
# ============================================================
print(f'\n\n{"="*70}')
print('PART 2: BCR Metrics — Chronic-Specific Analysis')
print('='*70)

bcr_rows = []
for (stage, tissue, donor), grp in obs.groupby(['Stage','tissue','donor'], observed=True):
    n = len(grp)
    bcr = grp[grp['BCR_clone.id'].notna()]
    nb = len(bcr)
    if nb == 0:
        bcr_rows.append({'Stage':stage,'tissue':tissue,'donor':donor,
                         'n_bcr':0,'clonality':np.nan,'pct_IgD':np.nan,
                         'pct_IgM':np.nan,'pct_switched':np.nan,'top_clone':0,
                         'pct_IGHV3_23':np.nan,'pct_bcr':0})
        continue
    cc=bcr['BCR_clone.id'].value_counts()
    iso=bcr['BCR_CType'].value_counts(); it=iso.sum()
    vg=bcr['BCR_v_gene'].value_counts()
    bcr_rows.append({'Stage':stage,'tissue':tissue,'donor':donor,
                     'n_bcr':nb,'clonality':safe_clonality(cc),
                     'pct_IgD':iso.get('IGHD',0)/it*100,
                     'pct_IgM':iso.get('IGHM',0)/it*100,
                     'pct_switched':(iso.get('IGHG',0)+iso.get('IGHA',0))/it*100,
                     'top_clone':cc.max(),
                     'pct_IGHV3_23':vg.get('IGHV3-23',0)/nb*100,
                     'pct_bcr':nb/n*100})
bcr_df = pd.DataFrame(bcr_rows)

key_bcr = [('Blood', 'pct_IgD', 'IgD% (naive B depletion)'),
           ('Blood', 'top_clone', 'Top clone size'),
           ('Blood', 'clonality', 'BCR clonality'),
           ('Liver', 'pct_IGHV3_23', 'IGHV3-23%')]

for tissue_val, metric, label in key_bcr:
    print(f'\n{"─"*70}')
    print(f'  {tissue_val} {label}')
    print(f'{"─"*70}')
    df = bcr_df[(bcr_df['tissue']==tissue_val) & (bcr_df['n_bcr']>0)]

    for stage in ['NL','IT','IA','AR']:
        s = df[df['Stage']==stage]
        if len(s)==0: continue
        vals = s[metric].values
        print(f'  {stage} (n={len(s)}): mean={s[metric].mean():.2f}, '
              f'donors=[{", ".join(f"{v:.2f}" for v in vals)}]')

    print(f'\n  Key comparisons:')
    for s1, s2 in [('NL','IT'),('NL','IA'),('IT','IA'),('IA','AR'),('IT','AR')]:
        a = df[df['Stage']==s1][metric]
        b = df[df['Stage']==s2][metric]
        r = mw(a, b)
        if r:
            print(f'    {r["sig"]} {s1}→{s2}: {r["A_m"]:.2f}→{r["B_m"]:.2f} '
                  f'({r["d"]}{abs(r["pct"]):.0f}%) p={r["p"]:.4f} [{r["cons"]}]')



PART 2: BCR Metrics — Chronic-Specific Analysis

──────────────────────────────────────────────────────────────────────
  Blood IgD% (naive B depletion)
──────────────────────────────────────────────────────────────────────
  NL (n=5): mean=17.71, donors=[23.08, 5.56, 4.44, 38.10, 17.39]
  IT (n=5): mean=2.23, donors=[4.51, 2.24, 1.05, 1.89, 1.47]
  IA (n=4): mean=2.66, donors=[3.02, 2.66, 2.83, 2.13]
  AR (n=1): mean=2.93, donors=[2.93]

  Key comparisons:
    ★ NL→IT: 17.71→2.23 (↓87%) p=0.0159 [24/25]
    ★ NL→IA: 17.71→2.66 (↓85%) p=0.0159 [20/20]
      IT→IA: 2.23→2.66 (↑19%) p=0.2857 [15/20]

──────────────────────────────────────────────────────────────────────
  Blood Top clone size
──────────────────────────────────────────────────────────────────────
  NL (n=5): mean=1.00, donors=[1.00, 1.00, 1.00, 1.00, 1.00]
  IT (n=5): mean=3.00, donors=[2.00, 2.00, 7.00, 2.00, 2.00]
  IA (n=4): mean=20.75, donors=[5.00, 71.00, 2.00, 5.00]
  AR (n=1): mean=2.00, donors=[2.00]

  Key comp

In [13]:
# ============================================================
# PART 3: TCR METRICS — Chronic-Specific Analysis
# ============================================================
print(f'\n\n{"="*70}')
print('PART 3: TCR Metrics — Chronic-Specific Analysis')
print('='*70)

tcr_rows = []
for (stage, tissue, donor), grp in obs.groupby(['Stage','tissue','donor'], observed=True):
    n = len(grp)
    tcr = grp[grp['TCR_clone.id'].notna()]
    nt = len(tcr)
    if nt == 0:
        tcr_rows.append({'Stage':stage,'tissue':tissue,'donor':donor,
                         'n_tcr':0,'clonality':np.nan,'pct_tcr':0})
        continue
    cc=tcr['TCR_clone.id'].value_counts()
    tcr_rows.append({'Stage':stage,'tissue':tissue,'donor':donor,
                     'n_tcr':nt,'clonality':safe_clonality(cc),'pct_tcr':nt/n*100})
tcr_df = pd.DataFrame(tcr_rows)

for tissue_val in ['Blood', 'Liver']:
    print(f'\n{"─"*70}')
    print(f'  {tissue_val} TCR Clonality')
    print(f'{"─"*70}')
    df = tcr_df[(tcr_df['tissue']==tissue_val) & (tcr_df['n_tcr']>0)]

    for stage in ['NL','IT','IA','AR']:
        s = df[df['Stage']==stage]
        if len(s)==0: continue
        print(f'  {stage} (n={len(s)}): mean={s.clonality.mean():.4f}, '
              f'donors=[{", ".join(f"{v:.4f}" for v in s.clonality.values)}]')

    print(f'\n  Key comparisons:')
    for s1, s2 in [('NL','IT'),('NL','IA'),('IT','IA'),('IA','AR'),('IT','AR')]:
        a = df[df['Stage']==s1]['clonality']
        b = df[df['Stage']==s2]['clonality']
        r = mw(a, b)
        if r:
            print(f'    {r["sig"]} {s1}→{s2}: {r["A_m"]:.4f}→{r["B_m"]:.4f} '
                  f'({r["d"]}{abs(r["pct"]):.0f}%) p={r["p"]:.4f} [{r["cons"]}]')



PART 3: TCR Metrics — Chronic-Specific Analysis

──────────────────────────────────────────────────────────────────────
  Blood TCR Clonality
──────────────────────────────────────────────────────────────────────
  NL (n=5): mean=0.5853, donors=[0.3404, 0.5586, 0.6465, 0.6532, 0.7277]
  IT (n=5): mean=0.3515, donors=[0.4893, 0.3993, 0.2511, 0.2921, 0.3256]
  IA (n=4): mean=0.3023, donors=[0.3586, 0.3483, 0.2397, 0.2627]
  AR (n=1): mean=0.3907, donors=[0.3907]

  Key comparisons:
    ★ NL→IT: 0.5853→0.3515 (↓40%) p=0.0317 [23/25]
    † NL→IA: 0.5853→0.3023 (↓48%) p=0.0635 [18/20]
      IT→IA: 0.3515→0.3023 (↓14%) p=0.5556 [13/20]

──────────────────────────────────────────────────────────────────────
  Liver TCR Clonality
──────────────────────────────────────────────────────────────────────
  NL (n=6): mean=0.5501, donors=[0.4679, 0.4631, 0.6523, 0.6130, 0.5850, 0.5195]
  IT (n=6): mean=0.4874, donors=[0.5139, 0.3620, 0.3741, 0.4566, 0.7658, 0.4522]
  IA (n=5): mean=0.3653, donors=[

In [14]:
# ============================================================
# PART 4: SYNTHESIS — Classification Table
# ============================================================
print(f'\n\n{"="*70}')
print('SYNTHESIS: Chronic HBV-Specific vs General Hepatitis Response')
print('='*70)
print('''
Classification logic:
  - NL→IT sig + IA≈AR → General hepatitis response (not chronic-specific)
  - NL→IT sig + IA≠AR or IT≠AR → Chronic HBV-specific
  - NL→IT sig + IA≈AR but IT≠AR → IT-specific (reversed at IA)

Key question for anti-HBs failure:
  Features present in IT/IA/CR but NOT in AR (which makes anti-HBs)
  = candidates for anti-HBs production blockade mechanism
''')

import os
SAVE_DIR = '/content/drive/MyDrive/ITLAS/results/version18-analysis-v2/BCR_TCR'
os.makedirs(SAVE_DIR, exist_ok=True)

# Save all pairwise results
bcr_df.to_csv(f'{SAVE_DIR}/C12c_BCR_donor_level_full.csv', index=False)
tcr_df.to_csv(f'{SAVE_DIR}/C12c_TCR_donor_level_full.csv', index=False)
sc_df.to_csv(f'{SAVE_DIR}/C12c_subcluster_proportions_full.csv', index=False)
print(f'\nSaved to: {SAVE_DIR}')



SYNTHESIS: Chronic HBV-Specific vs General Hepatitis Response

Classification logic:
  - NL→IT sig + IA≈AR → General hepatitis response (not chronic-specific)
  - NL→IT sig + IA≠AR or IT≠AR → Chronic HBV-specific
  - NL→IT sig + IA≈AR but IT≠AR → IT-specific (reversed at IA)

Key question for anti-HBs failure:
  Features present in IT/IA/CR but NOT in AR (which makes anti-HBs)
  = candidates for anti-HBs production blockade mechanism


Saved to: /content/drive/MyDrive/ITLAS/results/version18-analysis-v2/BCR_TCR


In [ ]:
# C12d: YoungMin's 3 specific hypothesis tests

In [15]:
from scipy.stats import mannwhitneyu
import numpy as np

def mw_print(a, b, labelA, labelB, name):
    a = np.array([x for x in a if not np.isnan(x)])
    b = np.array([x for x in b if not np.isnan(x)])
    if len(a)<2 or len(b)<2:
        print(f'  {name}: insufficient data (n={len(a)} vs {len(b)})')
        return
    stat,p = mannwhitneyu(a, b, alternative='two-sided')
    am,bm = a.mean(), b.mean()
    d = '↑' if bm>am else '↓'
    pct = ((bm-am)/am*100) if am!=0 else float('inf')
    pt=0;pc=0
    for av in a:
        for bv in b:
            pt+=1
            if (bm>am and bv>av) or (bm<=am and bv<=av): pc+=1
    sig = '★' if p<0.05 else '†' if p<0.10 else ' '
    print(f'  {sig} {labelA}(n={len(a)}, mean={am:.1f}%) vs {labelB}(n={len(b)}, mean={bm:.1f}%) '
          f'→ {d}{abs(pct):.0f}% p={p:.4f} [{pc}/{pt}]')


In [16]:
# ============================================================
# (2) FCRL5: (IT + IA excluding outlier 0.8%) vs (NL + AR)
# ============================================================
print('='*70)
print('(2) Liver B_c07-FCRL5: (IT+IA excl outlier) vs (NL+AR)')
print('='*70)

# From C12c output:
NL_fcrl5 = [22.6, 20.0, 13.1, 10.5, 6.9, 4.7]
IT_fcrl5 = [37.8, 37.7, 31.2, 18.5, 16.0]
IA_fcrl5 = [38.5, 31.3, 27.9, 21.3, 0.8]  # 0.8 = outlier
AR_fcrl5 = [24.5, 17.9, 5.9]

# IA without outlier
IA_fcrl5_no_outlier = [38.5, 31.3, 27.9, 21.3]

chronic_group = IT_fcrl5 + IA_fcrl5_no_outlier  # IT + IA (excl outlier)
non_chronic_group = NL_fcrl5 + AR_fcrl5  # NL + AR

print(f'\n  IT+IA(excl 0.8%): n={len(chronic_group)}, mean={np.mean(chronic_group):.1f}%')
print(f'    donors: {sorted(chronic_group, reverse=True)}')
print(f'  NL+AR: n={len(non_chronic_group)}, mean={np.mean(non_chronic_group):.1f}%')
print(f'    donors: {sorted(non_chronic_group, reverse=True)}')

mw_print(non_chronic_group, chronic_group, 'NL+AR', 'IT+IA(-outlier)', 'FCRL5')

# Also: with outlier included for comparison
chronic_with_outlier = IT_fcrl5 + IA_fcrl5
print(f'\n  For comparison — with IA outlier (0.8%) included:')
mw_print(NL_fcrl5 + AR_fcrl5, chronic_with_outlier, 'NL+AR', 'IT+IA(all)', 'FCRL5')

# And: IT alone vs NL+AR
print(f'\n  IT alone vs NL+AR:')
mw_print(NL_fcrl5 + AR_fcrl5, IT_fcrl5, 'NL+AR', 'IT', 'FCRL5')

(2) Liver B_c07-FCRL5: (IT+IA excl outlier) vs (NL+AR)

  IT+IA(excl 0.8%): n=9, mean=28.9%
    donors: [38.5, 37.8, 37.7, 31.3, 31.2, 27.9, 21.3, 18.5, 16.0]
  NL+AR: n=9, mean=14.0%
    donors: [24.5, 22.6, 20.0, 17.9, 13.1, 10.5, 6.9, 5.9, 4.7]
  ★ NL+AR(n=9, mean=14.0%) vs IT+IA(-outlier)(n=9, mean=28.9%) → ↑106% p=0.0062 [72/81]

  For comparison — with IA outlier (0.8%) included:
  ★ NL+AR(n=9, mean=14.0%) vs IT+IA(all)(n=10, mean=26.1%) → ↑86% p=0.0305 [72/90]

  IT alone vs NL+AR:
  ★ NL+AR(n=9, mean=14.0%) vs IT(n=5, mean=28.2%) → ↑102% p=0.0420 [38/45]


In [17]:
# ============================================================
# (3) MKI67: (IT+IA) vs NL alone
# ============================================================
print(f'\n{"="*70}')
print('(3) Liver plasmaB_c03-MKI67: (IT+IA) vs NL alone')
print('='*70)

NL_mki67 = [21.1, 16.2, 6.0, 4.1, 1.1, 0.7]
IT_mki67 = [4.0, 1.9, 0.7, 0.5, 0.0]
IA_mki67 = [1.9, 1.5, 0.7, 0.3, 0.0]

chronic_mki = IT_mki67 + IA_mki67
print(f'\n  IT+IA: n={len(chronic_mki)}, mean={np.mean(chronic_mki):.1f}%')
print(f'    donors: {sorted(chronic_mki, reverse=True)}')
print(f'  NL: n={len(NL_mki67)}, mean={np.mean(NL_mki67):.1f}%')
print(f'    donors: {sorted(NL_mki67, reverse=True)}')

mw_print(NL_mki67, chronic_mki, 'NL', 'IT+IA', 'MKI67')

# IT alone vs NL for reference
print(f'\n  For reference — IT alone vs NL:')
mw_print(NL_mki67, IT_mki67, 'NL', 'IT', 'MKI67')



(3) Liver plasmaB_c03-MKI67: (IT+IA) vs NL alone

  IT+IA: n=10, mean=1.1%
    donors: [4.0, 1.9, 1.9, 1.5, 0.7, 0.7, 0.5, 0.3, 0.0, 0.0]
  NL: n=6, mean=8.2%
    donors: [21.1, 16.2, 6.0, 4.1, 1.1, 0.7]
  ★ NL(n=6, mean=8.2%) vs IT+IA(n=10, mean=1.1%) → ↓86% p=0.0255 [52/60]

  For reference — IT alone vs NL:
  † NL(n=6, mean=8.2%) vs IT(n=5, mean=1.4%) → ↓83% p=0.0673 [26/30]


In [18]:
# ============================================================
# (1) B_c04-COCH: Biological role exploration
# ============================================================
print(f'\n{"="*70}')
print('(1) Liver B_c04-COCH: IT context analysis')
print('='*70)

NL_coch = [4.0, 1.7, 1.6, 0.6, 0.0, 0.0]
IT_coch = [12.1, 8.0, 6.3, 5.2, 3.7]
IA_coch = [29.4, 9.2, 8.3, 3.7, 2.9]
AR_coch = [18.5, 13.4, 6.4]

# IT vs AR (direct comparison)
print(f'\n  IT vs AR:')
mw_print(IT_coch, AR_coch, 'IT', 'AR', 'COCH')

# (IT+IA excl outlier 29.4) vs AR
IA_coch_no_outlier = [9.2, 8.3, 3.7, 2.9]
chronic_coch = IT_coch + IA_coch_no_outlier
print(f'\n  IT+IA(excl 29.4) vs AR:')
mw_print(chronic_coch, AR_coch, 'IT+IA(-outlier)', 'AR', 'COCH')

# Key: What proportion of COCH increase is "IT baseline" vs "hepatitis addition"?
print(f'\n  Baseline analysis:')
print(f'    NL mean: {np.mean(NL_coch):.1f}%')
print(f'    IT mean: {np.mean(IT_coch):.1f}% (IT "baseline" without hepatitis)')
print(f'    IA mean (excl outlier): {np.mean(IA_coch_no_outlier):.1f}% (hepatitis added)')
print(f'    AR mean: {np.mean(AR_coch):.1f}% (pure hepatitis response)')
print(f'    IT→IA increment: {np.mean(IA_coch_no_outlier) - np.mean(IT_coch):.1f}%')
print(f'    NL→AR increment: {np.mean(AR_coch) - np.mean(NL_coch):.1f}%')
print(f'    NL→IT increment: {np.mean(IT_coch) - np.mean(NL_coch):.1f}%')
print(f'    → IT "pre-hepatitis" COCH elevation = {(np.mean(IT_coch)-np.mean(NL_coch))/(np.mean(AR_coch)-np.mean(NL_coch))*100:.0f}% of AR hepatitis response')



(1) Liver B_c04-COCH: IT context analysis

  IT vs AR:
    IT(n=5, mean=7.1%) vs AR(n=3, mean=12.8%) → ↑81% p=0.1429 [13/15]

  IT+IA(excl 29.4) vs AR:
  † IT+IA(-outlier)(n=9, mean=6.6%) vs AR(n=3, mean=12.8%) → ↑93% p=0.0955 [23/27]

  Baseline analysis:
    NL mean: 1.3%
    IT mean: 7.1% (IT "baseline" without hepatitis)
    IA mean (excl outlier): 6.0% (hepatitis added)
    AR mean: 12.8% (pure hepatitis response)
    IT→IA increment: -1.0%
    NL→AR increment: 11.4%
    NL→IT increment: 5.7%
    → IT "pre-hepatitis" COCH elevation = 50% of AR hepatitis response
